# 真实 API：LLM-as-a-Judge 与评判团

用真实评判模型对同一回答输出 binary/continuous/Likert 三种裁决，并演示多评判官 panel 的平均风险。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai]"`；评判模型应与生成模型分离或至少独立调用。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `mode: str`：binary（忠实/幻觉+置信度）、continuous（0-100）、likert（1-5 忠实度）。
- `judge_name: str | None`：报告中的评判官名称。
- `LLMPanelScorer(judges)`：对多个独立 judge 的风险取平均。
- `TEMPERATURE: float = 0.8`：生成温度（评判本身用 temperature=0）。

## 步骤 1：配置真实 API

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

## 步骤 2：构建 judge 与 panel

In [ ]:
from hallucination.detection import DetectionExample
from hallucination.detection.judges import LLMJudgeScorer, LLMPanelScorer
judges = {mode: LLMJudgeScorer(backend, mode=mode) for mode in ("binary", "continuous", "likert")}
panel = LLMPanelScorer([LLMJudgeScorer(backend, mode="binary"), LLMJudgeScorer(backend, mode="likert")])

## 步骤 3：交互运行并分别读取

In [ ]:
prompt = input("Prompt：").strip()
response = input("待评回答（直接回车则由后端生成）：").strip() or backend.generate(prompt)
example = DetectionExample(prompt=prompt, response=response)
for mode, judge in judges.items():
    result = judge.score(example)
    print(mode, "| risk:", result.risk, "| parsed:", result.raw.get("parsed"))
panel_result = panel.score(example)
print("panel | risk:", panel_result.risk, "| judges:", panel_result.raw["num_judges"])

## 结果解读与限制

judge 输出是另一个模型的意见，不是事实概率；同一模型自评存在偏置，panel 平均也无法消除系统性偏差。